# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Research question:** Which content items in a client's portfolio are declining in search
visibility right now, and can a model rank them for review more usefully than a simple,
hand-written rule?

**The decision this supports:** out of thousands of pages a client owns, a human editor can
only manually review a handful per week. This work produces a ranked queue — which page to
look at *first* — not a diagnosis of *why* a page declined, and not an automated fix. It's
decision-support for a content strategist's weekly review, same framing as the Week-4 baseline
and Week-7 action playbook, now built on the full warehouse instead of the 30k-row starter slice.

**Why this matters at warehouse scale:** the starter dataset was one static snapshot. The
warehouse lets this become a real per-client, per-window question — "declining relative to
that client's own recent history" — rather than one global cutoff applied to everyone,
which is the more honest version of the same problem.

In [2]:
import duckdb, getpass, os

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF read token: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
dim_content = f"read_parquet('{REL}/dim_content.parquet')"
dim_clients = f"read_parquet('{REL}/dim_clients.parquet')"

# Scale check: how many clients and content items even exist, to size the "which page first" problem
scale = con.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM {dim_clients}) AS n_clients,
        (SELECT COUNT(*) FROM {dim_content}) AS n_content_items
""").df()
print(scale)
print("\nAt this scale, a human reviewing even 20 pages/week per client cannot cover the portfolio "
      "without a ranked queue -- that's the concrete case for this question.")

HF read token: ··········
   n_clients  n_content_items
0        104           519606

At this scale, a human reviewing even 20 pages/week per client cannot cover the portfolio without a ranked queue -- that's the concrete case for this question.


In [5]:
# Consolidate into one TABLES dict so every cell from here on uses the same reference,
# instead of separate loose variables per table.
TABLES = {
    'dim_clients':       dim_clients,
    'dim_content':       dim_content,
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_daily_full':   f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print(list(TABLES.keys()))

['dim_clients', 'dim_content', 'fact_daily_sample', 'fact_daily_full', 'fact_query_90d']


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [8]:
print("=== fact_content_query_90d ===")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_query_90d']}").df())

sample = con.sql(f"SELECT * FROM {TABLES['fact_query_90d']} LIMIT 5").df()
print(sample)

=== fact_content_query_90d ===
                      column_name column_type null   key default extra
0                  client_hash_id     VARCHAR  YES  None    None  None
1                 content_hash_id     VARCHAR  YES  None    None  None
2                   query_hash_id     VARCHAR  YES  None    None  None
3                query_char_count      BIGINT  YES  None    None  None
4               query_token_count      BIGINT  YES  None    None  None
5                    window_start        DATE  YES  None    None  None
6                      window_end        DATE  YES  None    None  None
7                 impressions_90d      BIGINT  YES  None    None  None
8                      clicks_90d      BIGINT  YES  None    None  None
9              impressions_last30      BIGINT  YES  None    None  None
10                  clicks_last30      BIGINT  YES  None    None  None
11             impressions_prev30      BIGINT  YES  None    None  None
12                  clicks_prev30      BIGINT 

In [12]:
window_check = con.sql(f"""
    SELECT COUNT(DISTINCT window_start) AS n_starts, COUNT(DISTINCT window_end) AS n_ends,
           MIN(window_start) AS min_start, MAX(window_end) AS max_end
    FROM {TABLES['fact_query_90d']}
""").df()
print(window_check)

   n_starts  n_ends  min_start    max_end
0         1       1 2026-04-02 2026-06-30


In [16]:
print("=== fact_content_daily_performance_sample ===")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily_sample']}").df())

print("\n=== dim_clients ===")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_clients']}").df())

print("\n=== dim_content ===")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df())

=== fact_content_daily_performance_sample ===
                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users  

In [17]:
def build_labeled_features(fact_table: str, min_prev30_impressions: int = 100):
    return con.sql(f"""
        WITH client_anchor AS (
            SELECT client_hash_id, MAX(report_date) AS anchor_date
            FROM {fact_table}
            WHERE gsc_data_available = TRUE
            GROUP BY client_hash_id
        ),
        eligible AS (
            SELECT ca.client_hash_id, ca.anchor_date
            FROM client_anchor ca
            JOIN {TABLES['dim_clients']} c ON c.client_hash_id = ca.client_hash_id
            WHERE c.gsc_data_start <= ca.anchor_date - INTERVAL 59 DAY
              AND c.is_active = TRUE
        ),
        windowed AS (
            SELECT
                f.client_hash_id,
                f.content_hash_id,
                e.anchor_date,
                SUM(CASE WHEN f.report_date > e.anchor_date - INTERVAL 30 DAY
                          AND f.gsc_data_available = TRUE
                         THEN f.gsc_impressions ELSE 0 END) AS impressions_last30,
                SUM(CASE WHEN f.report_date <= e.anchor_date - INTERVAL 30 DAY
                          AND f.report_date > e.anchor_date - INTERVAL 60 DAY
                          AND f.gsc_data_available = TRUE
                         THEN f.gsc_impressions ELSE 0 END) AS impressions_prev30,
                AVG(CASE WHEN f.report_date <= e.anchor_date - INTERVAL 30 DAY
                          AND f.report_date > e.anchor_date - INTERVAL 60 DAY
                          AND f.gsc_data_available = TRUE
                         THEN f.gsc_avg_position END) AS avg_position_prev30
            FROM {fact_table} f
            JOIN eligible e ON e.client_hash_id = f.client_hash_id
            WHERE f.report_date > e.anchor_date - INTERVAL 60 DAY
            GROUP BY 1, 2, 3
        )
        SELECT w.*,
            dc.content_created_date,
            dc.content_updated_date,
            (dc.content_created_date <= w.anchor_date - INTERVAL 59 DAY) AS content_old_enough,
            DATE_DIFF('day', dc.content_updated_date, w.anchor_date) AS days_since_last_update,
            (hash(w.client_hash_id) % 1000) < 250 AS is_test_client,
            CASE
                WHEN w.impressions_prev30 = 0 AND w.impressions_last30 = 0 THEN 'flat'
                WHEN w.impressions_prev30 = 0 AND w.impressions_last30 > 0 THEN 'new'
                WHEN (w.impressions_last30 - w.impressions_prev30) * 1.0 / NULLIF(w.impressions_prev30, 0) < -0.20 THEN 'down'
                WHEN (w.impressions_last30 - w.impressions_prev30) * 1.0 / NULLIF(w.impressions_prev30, 0) > 0.20 THEN 'up'
                ELSE 'stable'
            END AS trend_direction
        FROM windowed w
        JOIN {TABLES['dim_content']} dc ON dc.content_hash_id = w.content_hash_id
        WHERE w.impressions_prev30 >= {min_prev30_impressions}
          AND dc.is_deleted = FALSE
          AND dc.is_published = TRUE
    """).df()

features_dev = build_labeled_features(TABLES['fact_daily_sample'])
features_dev = features_dev[features_dev['content_old_enough']]
features_dev['is_declining_label'] = (features_dev['trend_direction'] == 'down').astype(int)

print(f"{len(features_dev):,} eligible rows")
print(features_dev['trend_direction'].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

0 eligible rows
Series([], Name: count, dtype: int64)


In [19]:
span = con.sql(f"SELECT MIN(report_date) AS min_d, MAX(report_date) AS max_d, COUNT(DISTINCT report_date) AS n_days FROM {TABLES['fact_daily_sample']}").df()
print(span)

       min_d      max_d  n_days
0 2026-06-01 2026-06-30      30


In [20]:
features_full = build_labeled_features(TABLES['fact_daily_full'])
features_full = features_full[features_full['content_old_enough']]
features_full['is_declining_label'] = (features_full['trend_direction'] == 'down').astype(int)

print(f"{len(features_full):,} eligible rows")
print(features_full['trend_direction'].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

93,401 eligible rows
trend_direction
down      62428
stable    16344
up        14629
Name: count, dtype: int64


In [21]:
final = features_full.merge(query_features, on=['client_hash_id', 'content_hash_id'], how='left')
print(f"{final['visible_queries'].notna().mean():.1%} of rows have aligned query-level features")

92.6% of rows have aligned query-level features


In [22]:
# Is the high decline rate concentrated among clients whose anchor_date is NOT the global max
# (i.e. they went quiet early -- more likely #2, an artifact) vs. clients still reporting
# right up to the table's true end (more likely #1, a real trend)?
anchor_dist = con.sql(f"""
    SELECT MAX(report_date) AS global_max_date FROM {TABLES['fact_daily_full']}
""").df()
print(anchor_dist)

features_full['anchor_is_global_max'] = features_full['anchor_date'] == anchor_dist['global_max_date'].iloc[0]
print(features_full.groupby('anchor_is_global_max')['is_declining_label'].agg(['mean', 'count']))

  global_max_date
0      2026-06-30
                          mean  count
anchor_is_global_max                 
False                 0.433333     30
True                  0.668462  93371


In [23]:
for floor in [100, 250, 500, 1000, 2500]:
    subset = features_full[features_full['impressions_prev30'] >= floor]
    print(f"floor={floor:>5}  n={len(subset):>6,}  decline_rate={subset['is_declining_label'].mean():.3f}")

floor=  100  n=93,401  decline_rate=0.668
floor=  250  n=68,870  decline_rate=0.669
floor=  500  n=51,088  decline_rate=0.669
floor= 1000  n=35,689  decline_rate=0.664
floor= 2500  n=19,614  decline_rate=0.658


In [24]:
full_span = con.sql(f"SELECT MIN(report_date) AS min_d, MAX(report_date) AS max_d, COUNT(DISTINCT client_hash_id) AS n_clients FROM {TABLES['fact_daily_full']}").df()
print(full_span)

       min_d      max_d  n_clients
0 2025-01-27 2026-06-30         70


In [26]:
# 1. Assemble the final feature table once
final = features_full.merge(query_features, on=['client_hash_id', 'content_hash_id'], how='left')

baseline_cols = ['impressions_prev30', 'avg_position_prev30', 'days_since_last_update']
query_cols = ['visible_queries', 'rare_share', 'anon_share', 'query_concentration_prev30']

y = final['is_declining_label']
is_test = final['is_test_client']

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

def evaluate(feature_cols, label):
    X = final[feature_cols]
    model = HistGradientBoostingClassifier(random_state=42).fit(X[~is_test], y[~is_test])
    probs = model.predict_proba(X[is_test])[:, 1]
    auc = roc_auc_score(y[is_test], probs)
    p20 = precision_at_k(probs, y[is_test].values, 20)
    print(f"{label:30} AUC={auc:.3f}  precision@20={p20:.3f}  n_features={len(feature_cols)}")
    return model, probs

import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

model_A, probs_A = evaluate(baseline_cols, "baseline features only")
model_B, probs_B = evaluate(baseline_cols + query_cols, "baseline + query features")

baseline features only         AUC=0.585  precision@20=0.800  n_features=3
baseline + query features      AUC=0.666  precision@20=1.000  n_features=7


In [27]:
safe_query_cols = ['query_impressions_prev30', 'query_concentration_prev30']  # prev30-only, safe
leaky_query_cols = ['visible_queries', 'rare_share', 'anon_share']            # full-90d-window, suspect

model_C, probs_C = evaluate(baseline_cols + safe_query_cols, "baseline + SAFE query features only")
model_D, probs_D = evaluate(baseline_cols + leaky_query_cols, "baseline + SUSPECT query features only")

baseline + SAFE query features only AUC=0.611  precision@20=0.850  n_features=5
baseline + SUSPECT query features only AUC=0.651  precision@20=1.000  n_features=6


In [28]:
final_feature_cols = baseline_cols + safe_query_cols  # ['impressions_prev30', 'avg_position_prev30',
                                                        #  'days_since_last_update', 'query_impressions_prev30',
                                                        #  'query_concentration_prev30']

model_final, probs_final = evaluate(final_feature_cols, "FINAL feature set (leakage-clean)")

FINAL feature set (leakage-clean) AUC=0.611  precision@20=0.850  n_features=5


**Tables used:**
- `fact_content_daily_performance` (full, 79M-row release) — daily grain, client × content × date.
  Development and iteration used `fact_content_daily_performance_sample` (single month,
  2026-06-01 to 2026-06-30) to validate query logic before running against the full table; the
  sample's single-month span cannot support a full prev30+last30 label pair per client, so all
  reported numbers here come from the full table, not the sample.
- `dim_clients` — client-level metadata, used for `gsc_data_start` (history-length gating) and
  `is_active`.
- `dim_content` — content-level metadata, used for `content_created_date`,
  `content_updated_date`, `is_published`, `is_deleted`.
- `fact_content_query_90d` — query-level detail, joined for concentration/rare-query features.
  This table has a single fixed window (`window_start = 2026-04-02`, `window_end = 2026-06-30`)
  rather than a per-content or per-client window, so its `_prev30` columns are only valid for
  content items whose independently-computed `anchor_date` equals `2026-06-30` exactly — using
  it elsewhere would leak part of a client's actual label window into a "prev30" feature.
  Coverage: 92.6% of eligible content items align; the remaining 7.4% carry `NULL`
  query-level features rather than an approximated value.

**Date windows:** anchored per client, not globally — `anchor_date = MAX(report_date)` within
each client's own `gsc_data_available = TRUE` rows (not one shared cutoff), because client
history start dates vary (`dim_clients.gsc_data_start`), making this an unbalanced panel.
`last30` = `(anchor_date - 29, anchor_date]`, `prev30` = `(anchor_date - 59, anchor_date - 30]`.

**Exclusions:**
- Content items where either the client (`gsc_data_start`) or the content itself
  (`content_created_date`) doesn't predate the prev30 window start by 59+ days — avoids
  mistaking "just started being tracked" for genuine decline.
- Days where `gsc_data_available = FALSE` — excluded from all sums rather than treated as
  real zero-traffic days.
- `is_deleted = TRUE` or `is_published = FALSE` content — removal isn't the same phenomenon
  as organic decline.
- Content items with fewer than 100 impressions in `prev30` — avoids computing a percent
  change off a near-zero, noise-dominated denominator.
- No client names, domains, URLs, or raw queries appear anywhere in this notebook or its
  outputs — every identifier is the warehouse's own anonymized hash.

**Eligible dataset:** 93,401 content items survive all filters. Base rate:
~67% `is_declining_label = 1` — confirmed stable (66.8%–65.8%) across impression-volume
floors from 100 to 2,500+, so this reflects a real pattern rather than small-number noise,
consistent with the FlyRank research paper's broader finding of industry-wide organic decline
as AI Overviews reshape search.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Assumptions:**
- `gsc_impressions` is treated as the visibility proxy — a real but imperfect stand-in for
  "how much this page is being seen in search," not a direct measure of business outcome
  (revenue, leads).
- A 20% impressions drop (last30 vs. prev30) is the decline threshold — arbitrary but
  consistent with the starter dataset's original label definition (`w02_ml_task_framing`),
  chosen for continuity rather than re-derived from scratch here.
- This dataset's 70 clients are assumed representative of FlyRank's broader client base for
  the purposes of this capstone — a small, single-platform sample, not a general claim about
  SEO behavior industry-wide.

**Label — `is_declining_label`:**  
Built per client, not against one global cutoff, because client history start dates vary
(`dim_clients.gsc_data_start`) — an unbalanced panel. For each client, `anchor_date =
MAX(report_date)` among their own `gsc_data_available = TRUE` rows. Two adjacent 30-day windows
follow: `last30 = (anchor_date - 29, anchor_date]` (label input only), `prev30 = (anchor_date -
59, anchor_date - 30]` (feature input only, never the label). `trend_pct = (impressions_last30 -
impressions_prev30) / impressions_prev30`; `is_declining_label = 1` when `trend_pct < -20%`.
Content items are excluded unless both the client and the content itself predate the prev30
window by 59+ days (avoids mistaking "just started tracking" for decline), and unless
`impressions_prev30 >= 100` (avoids computing a percent change off a near-zero denominator).

**Features (final, leakage-clean set):**
| Feature | Source | Window |
|---|---|---|
| `impressions_prev30` | daily fact, summed | prev30 only |
| `avg_position_prev30` | daily fact, averaged | prev30 only |
| `days_since_last_update` | `dim_content.content_updated_date` vs. anchor_date | static, pre-label |
| `query_impressions_prev30` | `fact_content_query_90d`, summed | prev30-scoped column, aligned subset only |
| `query_concentration_prev30` | `fact_content_query_90d`, Herfindahl-style | prev30-scoped, same aligned subset |


**Baseline rule**  
 (same design as `w04_baseline_score.ipynb`, transparent by construction —
no fitted weights): a content item is flagged if it (1) had real prior traffic
(`impressions_prev30 >= 500`), and either (2) hasn't been updated in 180+ days
(`stale`) or (3) sits at a worse-than-median `avg_position_prev30` (`slipping`). Score =
`visible × (stale + slipping) × impressions_prev30` — deliberately readable, not fitted.


**Validation design:**   
hash-based, per-client, fixed split — `hash(client_hash_id) % 1000
250` assigns ~25% of clients to test, independent of which table (sample or full) is queried
or which other clients are present. Chosen over `GroupShuffleSplit` specifically because it
guarantees the *same* clients land in test whether run against the 30-day sample or the full
17-month table — no re-splitting, no risk of a client crossing from train to test as the data
source scales up. With only 70 total clients, the ~17-18 test clients mean results carry
real group-level variance; treat single-run precision numbers as indicative, not exact.


**Leakage audit:**   
`fact_content_query_90d`'s `content_visible_query_count`,
`rare_impressions_share`, and `anonymized_impressions_share` are derived from
`content_total_impressions_90d` — the table's full, fixed 90-day window (2026-04-02 to
2026-06-30) — not a `_prev30`-scoped column. For the 92.6% of eligible content items whose
`anchor_date` equals the table's `window_end`, this window structurally contains their entire
last30 label period. Tested directly: these three columns alone lifted AUC from 0.585 to
0.651 and precision@20 from 0.80 to a suspicious 1.00 — nearly matching the full six-feature
result (AUC 0.666) — while the genuinely prev30-scoped pair (`query_impressions_prev30`,
`query_concentration_prev30`) produced only a modest, honest lift (AUC 0.611, precision@20
0.85). The three leaky columns were dropped. This mirrors the identical overlapping-window
mistake found in `w03_feature_leakage_check.ipynb` and `w06_validation_audit.ipynb`
(`log_impressions_90d`), now surfacing from a joined table instead of the original feature-prep
script — the same discipline caught it a second time.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.